# Koemi-3HIP HERM — T4 overnight training and measurement

This notebook is a complete, resumable experiment for the current repository. It follows the contracts in `README.md`, `docs/KOEMI_ARCHITECTURE.md`, `docs/BENCHMARK.md` and `MapSource.md`; it does not implement a second private training path.

The experiment uses the English `HuggingFaceTB/smol-smoltalk` training split. The dataset card describes 484,570 rows, Apache-2.0 licensing, shorter conversational samples for models below 1B parameters, and no advanced-math subset. The repository itself is byte-level, so every UTF-8 byte is one model position. Each record is normalized through the repository's `ShareGptRecordAdapter` and serialized with the repository's `<|system|>`, `<|input|>`, `<|thinking|>` and `<|output|>` contract.

The requested MoE setting is explicit: `expert_count=128`, `expert_top_k=6`. The router is a causal content hash over the current and previous byte, not a learned semantic router. Six expert updates are averaged and then normalized. This notebook measures that implementation; it does not claim semantic specialization or Transformer parity.

The run has a five-hour wall-clock budget. It saves optimizer, scaler, model, step counters and JSONL measurements to Google Drive when executed in Colab. Re-running the training cell resumes the checkpoint. All quality values use the correct token denominator; all speed values synchronize CUDA before timing.

In [ ]:
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
from collections import Counter
from dataclasses import asdict
from pathlib import Path

REPOSITORY_URL = "https://github.com/Koemi-AI/Koemi-3HIP.git"
REPOSITORY_DIR = Path("/content/Koemi-3HIP")
if not (REPOSITORY_DIR / "src" / "koemi").is_dir():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
sys.path.insert(0, str(REPOSITORY_DIR / "src"))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "datasets", "pandas", "matplotlib"],
    check=True,
)

try:
    from google.colab import drive
except ModuleNotFoundError:
    drive = None
if drive is not None:
    drive.mount("/content/drive", force_remount=False)
    RESULTS_DIR = Path("/content/drive/MyDrive/koemi-3hip-t4-results")
else:
    RESULTS_DIR = Path("/content/koemi-3hip-t4-results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset

from koemi.configuration.settings import ModelSettings, TrainingSettings
from koemi.data.adapters import ShareGptRecordAdapter
from koemi.data.contracts import DatasetValidationError
from koemi.data.serialization import serialize_record
from koemi.model.execution import ExecutionMode
from koemi.model.network import KoemiModel
from koemi.training.dataset import CausalByteDataset, IGNORE_TARGET_ID, create_training_loader
from koemi.training.objective import calculate_training_objective, token_cross_entropy

SEED = 1337
DATA_SEED = 20260913
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEVICE.type != "cuda":
    raise RuntimeError("This notebook requires a CUDA runtime with a T4-class GPU.")
GPU_PROPERTIES = torch.cuda.get_device_properties(DEVICE)
ENVIRONMENT = {
    "repository": str(REPOSITORY_DIR),
    "git_revision": subprocess.check_output(
        ["git", "-C", str(REPOSITORY_DIR), "rev-parse", "HEAD"], text=True
    ).strip(),
    "python": sys.version,
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(DEVICE),
    "gpu_total_memory_bytes": GPU_PROPERTIES.total_memory,
    "cuda_device_count": torch.cuda.device_count(),
}
(RESULTS_DIR / "environment.json").write_text(json.dumps(ENVIRONMENT, indent=2), encoding="utf-8")
print(json.dumps(ENVIRONMENT, indent=2))
print({"results_dir": str(RESULTS_DIR), "device": str(DEVICE)})

In [ ]:
DATASET_ID = "HuggingFaceTB/smol-smoltalk"
MAX_DATASET_RECORDS = 100_000
VALIDATION_FRACTION = 0.05
raw_dataset = load_dataset(DATASET_ID, split="train", streaming=True)
record_adapter = ShareGptRecordAdapter()
normalized_records = []
rejection_reasons = Counter()
source_counts = Counter()
for row_index, raw_row in enumerate(raw_dataset):
    if row_index >= MAX_DATASET_RECORDS:
        break
    try:
        messages = raw_row.get("messages")
        if not isinstance(messages, list):
            raise DatasetValidationError("row has no messages array")
        record = record_adapter.adapt(
            {"id": str(row_index), "conversations": messages}, str(row_index)
        )
        normalized_records.append(record)
        source_counts[str(raw_row.get("source", "unknown"))] += 1
    except Exception as error:
        rejection_reasons[type(error).__name__] += 1
if len(normalized_records) < 1_000:
    raise RuntimeError(
        f"Only {len(normalized_records)} valid records were loaded; "
        f"rejections={dict(rejection_reasons)}"
    )

split_rng = random.Random(DATA_SEED)
shuffled_indices = list(range(len(normalized_records)))
split_rng.shuffle(shuffled_indices)
validation_count = max(1, round(len(normalized_records) * VALIDATION_FRACTION))
validation_indices = set(shuffled_indices[:validation_count])
validation_records = tuple(
    record for index, record in enumerate(normalized_records) if index in validation_indices
)
training_records = tuple(
    record for index, record in enumerate(normalized_records) if index not in validation_indices
)

SEQUENCE_LENGTH = 256
training_dataset = CausalByteDataset(training_records, SEQUENCE_LENGTH)
validation_dataset = CausalByteDataset(validation_records, SEQUENCE_LENGTH)

def dataset_summary(dataset):
    input_tokens = sum(len(chunk.input_ids) for chunk in dataset.chunks)
    supervised_tokens = sum(
        sum(target_id != IGNORE_TARGET_ID for target_id in chunk.target_ids)
        for chunk in dataset.chunks
    )
    thinking_tokens = sum(
        sum(
            target_id != IGNORE_TARGET_ID and thinking
            for target_id, thinking in zip(chunk.target_ids, chunk.thinking_mask, strict=True)
        )
        for chunk in dataset.chunks
    )
    return {
        "chunks": len(dataset),
        "input_tokens": input_tokens,
        "supervised_tokens": supervised_tokens,
        "thinking_tokens": thinking_tokens,
    }

serialized_bytes = [len(serialize_record(record).token_bytes) for record in normalized_records]
DATASET_REPORT = {
    "dataset_id": DATASET_ID,
    "dataset_card_url": "https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk",
    "requested_rows": MAX_DATASET_RECORDS,
    "loaded_records": len(normalized_records),
    "training_records": len(training_records),
    "validation_records": len(validation_records),
    "validation_fraction": VALIDATION_FRACTION,
    "sequence_length": SEQUENCE_LENGTH,
    "mean_serialized_bytes": float(np.mean(serialized_bytes)),
    "p50_serialized_bytes": float(np.median(serialized_bytes)),
    "p95_serialized_bytes": float(np.percentile(serialized_bytes, 95)),
    "sources": dict(source_counts),
    "rejection_reasons": dict(rejection_reasons),
    "training_dataset": dataset_summary(training_dataset),
    "validation_dataset": dataset_summary(validation_dataset),
}
(RESULTS_DIR / "dataset.json").write_text(json.dumps(DATASET_REPORT, indent=2), encoding="utf-8")
print(json.dumps(DATASET_REPORT, indent=2))

In [ ]:
MODEL_SETTINGS = ModelSettings(
    embedding_size=128,
    memory_features=16,
    local_memory_size=16,
    salience_memory_size=16,
    salience_threshold=0.75,
    expert_count=128,
    expert_top_k=6,
    cache_capacity=256,
    scan_chunk=128,
    refine_decay_rate=0.0625,
    ablation="no_refine",
)
model = KoemiModel(MODEL_SETTINGS).to(DEVICE)
AMP_DTYPE = torch.float16
MAX_CANDIDATE_BATCH_SIZE = 16
MIN_CANDIDATE_BATCH_SIZE = 1
CANDIDATE_BATCH_SIZES = [
    value
    for value in (16, 8, 4, 2, 1)
    if MIN_CANDIDATE_BATCH_SIZE <= value <= MAX_CANDIDATE_BATCH_SIZE
]

def parameter_summary(module):
    parameters = list(module.parameters())
    return {
        "parameters": sum(parameter.numel() for parameter in parameters),
        "trainable_parameters": sum(parameter.numel() for parameter in parameters if parameter.requires_grad),
        "parameter_bytes_fp32": sum(parameter.numel() * 4 for parameter in parameters),
        "modules": sum(1 for _ in module.modules()),
    }

def state_summary(state):
    tensors = {
        "working_state": state.working_state,
        "memory_basis": state.memory_basis,
        "memory_normalizer": state.memory_normalizer,
        "refine_basis": state.refine_basis,
        "refine_normalizer": state.refine_normalizer,
        "local_keys": state.local_keys,
        "local_values": state.local_values,
        "local_valid": state.local_valid,
        "salient_keys": state.salient_keys,
        "salient_values": state.salient_values,
        "salient_valid": state.salient_valid,
        "last_token_ids": state.last_token_ids,
    }
    return {
        "total_bytes": sum(tensor.numel() * tensor.element_size() for tensor in tensors.values()),
        "tensor_bytes": {
            name: tensor.numel() * tensor.element_size() for name, tensor in tensors.items()
        },
        "shapes": {name: list(tensor.shape) for name, tensor in tensors.items()},
    }

def synchronize_cuda():
    torch.cuda.synchronize(DEVICE)

model.eval()
batch_probe_results = []
safe_batch_size = None
for candidate_batch_size in CANDIDATE_BATCH_SIZES:
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE)
    model.zero_grad(set_to_none=True)
    probe_input = torch.randint(
        0, 256, (candidate_batch_size, MODEL_SETTINGS.scan_chunk), device=DEVICE
    )
    probe_status = "ok"
    error_message = None
    probe_output = None
    probe_loss = None
    try:
        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
            probe_output = model(probe_input, execution_mode=ExecutionMode.PARALLEL)
            probe_loss = probe_output.logits.float().square().mean()
        probe_loss.backward()
        synchronize_cuda()
    except RuntimeError as error:
        if "out of memory" not in str(error).lower():
            raise
        probe_status = "oom"
        error_message = str(error).splitlines()[0]
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()
    peak_bytes = torch.cuda.max_memory_allocated(DEVICE)
    batch_probe_results.append(
        {
            "batch_size": candidate_batch_size,
            "status": probe_status,
            "peak_memory_bytes": peak_bytes,
            "error": error_message,
        }
    )
    del probe_input, probe_output, probe_loss
    if probe_status == "ok":
        safe_batch_size = candidate_batch_size
        break
model.zero_grad(set_to_none=True)
if safe_batch_size is None:
    raise RuntimeError(f"No candidate batch size fit the GPU: {batch_probe_results}")
state_shape_probe = torch.randint(0, 256, (1, max(32, MODEL_SETTINGS.scan_chunk)), device=DEVICE)
with torch.no_grad():
    state_capacity_state = model(state_shape_probe, execution_mode=ExecutionMode.PARALLEL).state
del state_shape_probe

BATCH_SIZE = safe_batch_size
NUM_WORKERS = min(2, os.cpu_count() or 1)
TRAIN_GENERATOR = torch.Generator().manual_seed(DATA_SEED)
train_loader = create_training_loader(
    training_dataset,
    BATCH_SIZE,
    generator=TRAIN_GENERATOR,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2,
)
validation_loader = create_training_loader(
    validation_dataset,
    BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2,
)
TARGET_EFFECTIVE_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = max(1, math.ceil(TARGET_EFFECTIVE_BATCH_SIZE / BATCH_SIZE))
TRAINING_SETTINGS = TrainingSettings(
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    epochs=1,
    learning_rate=3e-4,
    gradient_clip_norm=1.0,
    device="cuda",
    execution_mode="parallel",
    thinking_loss_weight=1.0,
    weight_decay=0.01,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=500,
    precision="fp16",
    label_smoothing=0.0,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    prefetch_factor=2,
)
OPTIMIZER = torch.optim.AdamW(
    model.parameters(),
    lr=TRAINING_SETTINGS.learning_rate,
    weight_decay=TRAINING_SETTINGS.weight_decay,
    betas=(0.9, 0.95),
)
SCHEDULE_STEPS = 100_000
SCHEDULER = torch.optim.lr_scheduler.LambdaLR(
    OPTIMIZER,
    lambda step: (
        min(1.0, (step + 1) / TRAINING_SETTINGS.warmup_steps)
        if TRAINING_SETTINGS.warmup_steps and step < TRAINING_SETTINGS.warmup_steps
        else 0.5
        * (1.0 + math.cos(math.pi * min(1.0, max(0.0, (step - TRAINING_SETTINGS.warmup_steps) / max(1, SCHEDULE_STEPS - TRAINING_SETTINGS.warmup_steps)))))
    ),
)
GRAD_SCALER = torch.amp.GradScaler("cuda", enabled=True)
MODEL_REPORT = {
    "settings": MODEL_SETTINGS.to_dict(),
    "training_settings": TRAINING_SETTINGS.to_dict(),
    "parameters": parameter_summary(model),
    "state_at_capacity_probe": state_summary(state_capacity_state),
    "batch_probe": batch_probe_results,
    "selected_batch_size": BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "effective_batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    "amp_dtype": str(AMP_DTYPE),
}
(RESULTS_DIR / "model.json").write_text(json.dumps(MODEL_REPORT, indent=2), encoding="utf-8")
print(json.dumps(MODEL_REPORT, indent=2))
del state_capacity_state

In [ ]:
def state_max_abs_difference(left_state, right_state):
    differences = {}
    for field_name in (
        "working_state", "memory_basis", "memory_normalizer", "refine_basis",
        "refine_normalizer", "local_keys", "local_values", "local_valid",
        "salient_keys", "salient_values", "salient_valid", "last_token_ids",
    ):
        left_value = getattr(left_state, field_name)
        right_value = getattr(right_state, field_name)
        if left_value.dtype == torch.bool or left_value.dtype == torch.long:
            differences[field_name] = 0.0 if torch.equal(left_value, right_value) else float("inf")
        else:
            differences[field_name] = float((left_value - right_value).abs().max().detach().cpu())
    differences["step_index"] = 0.0 if left_state.step_index == right_state.step_index else float("inf")
    return differences

model.eval()
contract_input = torch.randint(0, 256, (2, 37), device=DEVICE)
with torch.no_grad():
    with torch.autocast(device_type="cuda", enabled=False):
        parallel_output = model(contract_input, execution_mode=ExecutionMode.PARALLEL)
        sequential_output = model(contract_input, execution_mode=ExecutionMode.SEQUENTIAL)
parallel_sequential_report = {
    "logits_max_abs": float((parallel_output.logits - sequential_output.logits).abs().max().cpu()),
    "surprise_max_abs": float((parallel_output.surprise_values - sequential_output.surprise_values).abs().max().cpu()),
    "state_max_abs": state_max_abs_difference(parallel_output.state, sequential_output.state),
    "expert_indices_equal": bool(torch.equal(parallel_output.expert_indices, sequential_output.expert_indices)),
    "active_expert_indices_equal": bool(torch.equal(parallel_output.active_expert_indices, sequential_output.active_expert_indices)),
    "parallel_tokens": parallel_output.token_count,
    "sequential_tokens": sequential_output.token_count,
}
if parallel_sequential_report["logits_max_abs"] > 1e-3:
    raise AssertionError(f"parallel/sequential logits contract failed: {parallel_sequential_report}")
if not parallel_sequential_report["active_expert_indices_equal"]:
    raise AssertionError("parallel/sequential expert assignment contract failed")

confidence_numerator = torch.ones(1, MODEL_SETTINGS.embedding_size, device=DEVICE)
confidence_denominator = torch.zeros(1, 1, device=DEVICE)
confidence_read = model.associative_memory.confidence_weighted_read(
    confidence_numerator, confidence_denominator
)
if not torch.equal(confidence_read, torch.zeros_like(confidence_read)):
    raise AssertionError("zero-evidence associative reads must be zero")

valid_assignments = parallel_output.active_expert_indices[parallel_output.valid_positions]
if valid_assignments.shape[-1] != MODEL_SETTINGS.expert_top_k:
    raise AssertionError(f"expected top-k={MODEL_SETTINGS.expert_top_k}, got {valid_assignments.shape}")
if not bool((valid_assignments >= 0).all()):
    raise AssertionError("valid tokens received an unassigned expert")
sorted_assignments = valid_assignments.sort(dim=-1).values
if not bool((sorted_assignments[..., 1:] != sorted_assignments[..., :-1]).all()):
    raise AssertionError("top-k assignments are not distinct within a token")

PREFLIGHT_REPORT = {
    "parallel_sequential": parallel_sequential_report,
    "zero_evidence_read_max_abs": float(confidence_read.abs().max().cpu()),
    "top_k": MODEL_SETTINGS.expert_top_k,
    "valid_tokens_in_probe": int(parallel_output.valid_positions.sum().cpu()),
    "active_assignments_in_probe": int(valid_assignments.numel()),
    "fixed_state_bytes": MODEL_REPORT["state_at_capacity_probe"]["total_bytes"],
}
(RESULTS_DIR / "preflight.json").write_text(json.dumps(PREFLIGHT_REPORT, indent=2), encoding="utf-8")
print(json.dumps(PREFLIGHT_REPORT, indent=2))

In [ ]:
CHECKPOINT_PATH = RESULTS_DIR / "koemi-3hip-t4-moe-training.pt"
STEP_LOG_PATH = RESULTS_DIR / "training_steps.jsonl"
EPOCH_LOG_PATH = RESULTS_DIR / "training_epochs.jsonl"
CHECKPOINT_VERSION = 2
FIVE_HOUR_BUDGET_SECONDS = 5 * 60 * 60
CHECKPOINT_EVERY_STEPS = 250
LOG_FLUSH_EVERY_STEPS = 10

optimizer_steps = 0
epoch_index = 0
tokens_seen = 0
history_buffer = []
resumed_from_checkpoint = False
if STEP_LOG_PATH.exists() and not CHECKPOINT_PATH.exists():
    raise RuntimeError("training_steps.jsonl exists without its checkpoint; move the stale result directory before restarting")
if CHECKPOINT_PATH.exists():
    checkpoint_payload = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    if checkpoint_payload.get("checkpoint_version") != CHECKPOINT_VERSION:
        raise ValueError("checkpoint version does not match this notebook")
    if checkpoint_payload.get("model_settings") != MODEL_SETTINGS.to_dict():
        raise ValueError("checkpoint model settings do not match the active experiment")
    model.load_state_dict(checkpoint_payload["model_state"])
    OPTIMIZER.load_state_dict(checkpoint_payload["optimizer_state"])
    SCHEDULER.load_state_dict(checkpoint_payload["scheduler_state"])
    GRAD_SCALER.load_state_dict(checkpoint_payload["scaler_state"])
    optimizer_steps = int(checkpoint_payload["optimizer_steps"])
    epoch_index = int(checkpoint_payload["epoch_index"])
    tokens_seen = int(checkpoint_payload["tokens_seen"])
    if "train_generator_state" in checkpoint_payload:
        TRAIN_GENERATOR.set_state(checkpoint_payload["train_generator_state"])
    resumed_from_checkpoint = True
    print({"resumed": True, "optimizer_steps": optimizer_steps, "tokens_seen": tokens_seen})

def save_training_checkpoint():
    torch.save(
        {
            "checkpoint_version": CHECKPOINT_VERSION,
            "model_settings": MODEL_SETTINGS.to_dict(),
            "model_state": model.state_dict(),
            "optimizer_state": OPTIMIZER.state_dict(),
            "scheduler_state": SCHEDULER.state_dict(),
            "scaler_state": GRAD_SCALER.state_dict(),
            "optimizer_steps": optimizer_steps,
            "epoch_index": epoch_index,
            "tokens_seen": tokens_seen,
            "train_generator_state": TRAIN_GENERATOR.get_state(),
        },
        CHECKPOINT_PATH,
    )

def flush_step_log():
    if not history_buffer:
        return
    with STEP_LOG_PATH.open("a", encoding="utf-8") as log_file:
        for record in history_buffer:
            log_file.write(json.dumps(record) + "\n")
    history_buffer.clear()

def update_accumulator(accumulator, output, objective, supervised_count, elapsed_seconds):
    accumulator["objective_loss_sum"] += float(objective.total_loss.detach()) * supervised_count
    accumulator["task_loss_sum"] += float(objective.task_loss.detach()) * supervised_count
    accumulator["thinking_loss_sum"] += float(objective.thinking_loss.detach()) * supervised_count
    accumulator["supervised_tokens"] += supervised_count
    accumulator["valid_tokens"] += output.token_count
    accumulator["surprise_sum"] += float(
        output.surprise_values.masked_select(output.valid_positions).sum().detach()
    )
    accumulator["elapsed_seconds"] += elapsed_seconds
    if output.active_expert_indices is not None:
        assignments = output.active_expert_indices[output.valid_positions].reshape(-1)
        accumulator["expert_counts"] += torch.bincount(
            assignments, minlength=MODEL_SETTINGS.expert_count
        ).detach().cpu().numpy()

def finish_optimizer_step(accumulator, accumulated_batches, started_at):
    global optimizer_steps, tokens_seen
    GRAD_SCALER.unscale_(OPTIMIZER)
    if accumulated_batches < GRADIENT_ACCUMULATION_STEPS:
        correction = GRADIENT_ACCUMULATION_STEPS / accumulated_batches
        for parameter in model.parameters():
            if parameter.grad is not None:
                parameter.grad.mul_(correction)
    gradient_norm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0).detach().cpu())
    nonfinite_gradient = not math.isfinite(gradient_norm)
    synchronize_cuda()
    GRAD_SCALER.step(OPTIMIZER)
    GRAD_SCALER.update()
    SCHEDULER.step()
    synchronize_cuda()
    optimizer_steps += 1
    step_seconds = time.perf_counter() - started_at
    supervised_tokens = accumulator["supervised_tokens"]
    tokens_seen += supervised_tokens
    record = {
        "step": optimizer_steps,
        "epoch": epoch_index,
        "loss_nats": accumulator["objective_loss_sum"] / max(1, supervised_tokens),
        "task_loss_nats": accumulator["task_loss_sum"] / max(1, supervised_tokens),
        "thinking_loss_nats": accumulator["thinking_loss_sum"] / max(1, supervised_tokens),
        "bpb": accumulator["objective_loss_sum"] / max(1, supervised_tokens) / math.log(2),
        "supervised_tokens": supervised_tokens,
        "valid_tokens": accumulator["valid_tokens"],
        "tokens_seen": tokens_seen,
        "tokens_per_second": supervised_tokens / max(step_seconds, 1e-9),
        "step_seconds": step_seconds,
        "gradient_norm": gradient_norm,
        "nonfinite_gradient": nonfinite_gradient,
        "learning_rate": OPTIMIZER.param_groups[0]["lr"],
        "scaler_scale": float(GRAD_SCALER.get_scale()),
        "gpu_allocated_bytes": torch.cuda.memory_allocated(DEVICE),
        "gpu_reserved_bytes": torch.cuda.memory_reserved(DEVICE),
        "gpu_peak_allocated_bytes": torch.cuda.max_memory_allocated(DEVICE),
        "surprise_mean": accumulator["surprise_sum"] / max(1, accumulator["valid_tokens"]),
        "expert_counts": accumulator["expert_counts"].astype(int).tolist(),
    }
    history_buffer.append(record)
    OPTIMIZER.zero_grad(set_to_none=True)
    return record

model.train()
optimizer_started_at = None
accumulated_batches = 0
accumulation = {
    "objective_loss_sum": 0.0,
    "task_loss_sum": 0.0,
    "thinking_loss_sum": 0.0,
    "supervised_tokens": 0,
    "valid_tokens": 0,
    "surprise_sum": 0.0,
    "elapsed_seconds": 0.0,
    "expert_counts": np.zeros(MODEL_SETTINGS.expert_count, dtype=np.int64),
}
torch.cuda.reset_peak_memory_stats(DEVICE)
training_started_at = time.perf_counter()
completed_epochs = 0
try:
    while time.perf_counter() - training_started_at < FIVE_HOUR_BUDGET_SECONDS:
        epoch_index += 1
        epoch_started_at = time.perf_counter()
        epoch_accumulator = {
            "objective_loss_sum": 0.0,
            "task_loss_sum": 0.0,
            "thinking_loss_sum": 0.0,
            "supervised_tokens": 0,
            "valid_tokens": 0,
            "surprise_sum": 0.0,
            "elapsed_seconds": 0.0,
            "expert_counts": np.zeros(MODEL_SETTINGS.expert_count, dtype=np.int64),
        }
        supervised_batches = 0
        for batch in train_loader:
            if time.perf_counter() - training_started_at >= FIVE_HOUR_BUDGET_SECONDS:
                break
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            target_ids = batch["target_ids"].to(DEVICE, non_blocking=True)
            thinking_mask = batch["thinking_mask"].to(DEVICE, non_blocking=True)
            supervised_count = int((target_ids != IGNORE_TARGET_ID).sum().item())
            if supervised_count == 0:
                continue
            if optimizer_started_at is None:
                optimizer_started_at = time.perf_counter()
            batch_started_at = time.perf_counter()
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                output = model(input_ids, execution_mode=ExecutionMode.PARALLEL)
                objective = calculate_training_objective(
                    output,
                    target_ids,
                    thinking_mask,
                    TRAINING_SETTINGS.thinking_loss_weight,
                    TRAINING_SETTINGS.label_smoothing,
                )
                scaled_loss = objective.total_loss / GRADIENT_ACCUMULATION_STEPS
            GRAD_SCALER.scale(scaled_loss).backward()
            synchronize_cuda()
            batch_elapsed_seconds = time.perf_counter() - batch_started_at
            update_accumulator(
                accumulation, output, objective, supervised_count, batch_elapsed_seconds
            )
            update_accumulator(
                epoch_accumulator, output, objective, supervised_count, batch_elapsed_seconds
            )
            accumulated_batches += 1
            supervised_batches += 1
            if accumulated_batches == GRADIENT_ACCUMULATION_STEPS:
                record = finish_optimizer_step(
                    accumulation, accumulated_batches, optimizer_started_at
                )
                accumulation = {
                    "objective_loss_sum": 0.0,
                    "task_loss_sum": 0.0,
                    "thinking_loss_sum": 0.0,
                    "supervised_tokens": 0,
                    "valid_tokens": 0,
                    "surprise_sum": 0.0,
                    "elapsed_seconds": 0.0,
                    "expert_counts": np.zeros(MODEL_SETTINGS.expert_count, dtype=np.int64),
                }
                accumulated_batches = 0
                optimizer_started_at = None
                if optimizer_steps % LOG_FLUSH_EVERY_STEPS == 0:
                    flush_step_log()
                if optimizer_steps % CHECKPOINT_EVERY_STEPS == 0:
                    flush_step_log()
                    save_training_checkpoint()
                    print(record)
        if accumulated_batches > 0 and optimizer_started_at is not None:
            finish_optimizer_step(accumulation, accumulated_batches, optimizer_started_at)
            accumulation = {
                "objective_loss_sum": 0.0,
                "task_loss_sum": 0.0,
                "thinking_loss_sum": 0.0,
                "supervised_tokens": 0,
                "valid_tokens": 0,
                "surprise_sum": 0.0,
                "elapsed_seconds": 0.0,
                "expert_counts": np.zeros(MODEL_SETTINGS.expert_count, dtype=np.int64),
            }
            accumulated_batches = 0
            optimizer_started_at = None
        if supervised_batches == 0:
            raise RuntimeError("an entire training epoch produced no supervised batches")
        epoch_record = {
            "epoch": epoch_index,
            "optimizer_steps": optimizer_steps,
            "loss_nats": epoch_accumulator["objective_loss_sum"] / epoch_accumulator["supervised_tokens"],
            "task_loss_nats": epoch_accumulator["task_loss_sum"] / epoch_accumulator["supervised_tokens"],
            "thinking_loss_nats": epoch_accumulator["thinking_loss_sum"] / epoch_accumulator["supervised_tokens"],
            "bpb": epoch_accumulator["objective_loss_sum"] / epoch_accumulator["supervised_tokens"] / math.log(2),
            "supervised_tokens": epoch_accumulator["supervised_tokens"],
            "valid_tokens": epoch_accumulator["valid_tokens"],
            "elapsed_seconds": time.perf_counter() - epoch_started_at,
            "expert_counts": epoch_accumulator["expert_counts"].astype(int).tolist(),
        }
        with EPOCH_LOG_PATH.open("a", encoding="utf-8") as epoch_log:
            epoch_log.write(json.dumps(epoch_record) + "\n")
        completed_epochs += 1
        print(epoch_record)
except KeyboardInterrupt:
    print("Training interrupted by the notebook user; the finally block will save a resumable checkpoint.")
finally:
    flush_step_log()
    save_training_checkpoint()
TRAINING_SESSION = {
    "completed_epochs_this_session": completed_epochs,
    "optimizer_steps": optimizer_steps,
    "tokens_seen": tokens_seen,
    "session_wall_seconds": time.perf_counter() - training_started_at,
    "resumed_from_checkpoint": resumed_from_checkpoint,
    "checkpoint": str(CHECKPOINT_PATH),
    "step_log": str(STEP_LOG_PATH),
    "epoch_log": str(EPOCH_LOG_PATH),
}
(RESULTS_DIR / "training_session.json").write_text(
    json.dumps(TRAINING_SESSION, indent=2), encoding="utf-8"
)
print(json.dumps(TRAINING_SESSION, indent=2))

In [ ]:
def gini_coefficient(counts):
    values = np.asarray(counts, dtype=np.float64)
    total = values.sum()
    if total == 0:
        return 0.0
    return float(np.abs(np.subtract.outer(values, values)).sum() / (2 * len(values) * total))

def expert_load_summary(counts):
    values = np.asarray(counts, dtype=np.int64)
    total = int(values.sum())
    probabilities = values / max(1, total)
    occupied = probabilities > 0
    entropy = float(-(probabilities[occupied] * np.log(probabilities[occupied])).sum())
    return {
        "counts": values.astype(int).tolist(),
        "total_assignments": total,
        "occupied_experts": int(occupied.sum()),
        "entropy_nats": entropy,
        "normalized_entropy": entropy / math.log(len(values)),
        "load_min": int(values.min()),
        "load_max": int(values.max()),
        "load_mean": float(values.mean()),
        "load_gini": gini_coefficient(values),
    }

def online_statistics(sum_value, sum_square, count):
    if count == 0:
        return {"mean": None, "standard_error": None}
    mean_value = sum_value / count
    variance = max(0.0, sum_square / count - mean_value * mean_value)
    return {"mean": mean_value, "standard_error": math.sqrt(variance / count)}

def evaluate_loader(data_loader, split_name, maximum_batches=None):
    model.eval()
    total_loss_sum = 0.0
    task_loss_sum = 0.0
    answer_loss_sum = 0.0
    thinking_loss_sum = 0.0
    supervised_count = 0
    answer_count = 0
    thinking_count = 0
    valid_count = 0
    batch_times = []
    surprise_values = []
    expert_counts = np.zeros(MODEL_SETTINGS.expert_count, dtype=np.int64)
    state_norms = {name: [] for name in ("working", "memory", "refine", "local", "salient")}
    loss_square_sum = 0.0
    answer_loss_square_sum = 0.0
    nonfinite_batches = 0
    with torch.inference_mode():
        for batch_index, batch in enumerate(data_loader):
            if maximum_batches is not None and batch_index >= maximum_batches:
                break
            input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
            target_ids = batch["target_ids"].to(DEVICE, non_blocking=True)
            thinking_mask = batch["thinking_mask"].to(DEVICE, non_blocking=True)
            supervised_mask = target_ids != IGNORE_TARGET_ID
            batch_supervised = int(supervised_mask.sum().item())
            if batch_supervised == 0:
                continue
            synchronize_cuda()
            batch_started_at = time.perf_counter()
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                output = model(input_ids, execution_mode=ExecutionMode.PARALLEL)
            synchronize_cuda()
            batch_times.append(time.perf_counter() - batch_started_at)
            token_losses = token_cross_entropy(output.logits.float(), target_ids)
            answer_mask = supervised_mask & ~thinking_mask
            thinking_positions = supervised_mask & thinking_mask
            answer_tokens = int(answer_mask.sum().item())
            thinking_tokens = int(thinking_positions.sum().item())
            total_loss_sum += float((token_losses * supervised_mask).sum().cpu())
            task_loss_sum += float((token_losses * supervised_mask).sum().cpu())
            loss_square_sum += float((token_losses.square() * supervised_mask).sum().cpu())
            if answer_tokens:
                answer_loss_sum += float((token_losses * answer_mask).sum().cpu())
                answer_loss_square_sum += float((token_losses.square() * answer_mask).sum().cpu())
            if thinking_tokens:
                thinking_loss_sum += float((token_losses * thinking_positions).sum().cpu())
            supervised_count += batch_supervised
            answer_count += answer_tokens
            thinking_count += thinking_tokens
            valid_count += output.token_count
            if not bool(torch.isfinite(token_losses.masked_select(supervised_mask)).all()):
                nonfinite_batches += 1
            surprise_tensor = output.surprise_values.masked_select(output.valid_positions).float().cpu()
            remaining = max(0, 100_000 - len(surprise_values))
            if remaining:
                surprise_values.extend(surprise_tensor[:remaining].tolist())
            if output.active_expert_indices is not None:
                assignments = output.active_expert_indices[output.valid_positions].reshape(-1)
                expert_counts += torch.bincount(
                    assignments, minlength=MODEL_SETTINGS.expert_count
                ).cpu().numpy()
            state_norms["working"].append(float(output.state.working_state.norm(dim=-1).mean().cpu()))
            state_norms["memory"].append(float(output.state.memory_basis.norm(dim=(-2, -1)).mean().cpu()))
            state_norms["refine"].append(float(output.state.refine_basis.norm(dim=(-2, -1)).mean().cpu()))
            state_norms["local"].append(float(output.state.local_keys.norm(dim=(-2, -1)).mean().cpu()) if output.state.local_keys.numel() else 0.0)
            state_norms["salient"].append(float(output.state.salient_keys.norm(dim=(-2, -1)).mean().cpu()) if output.state.salient_keys.numel() else 0.0)
    if supervised_count == 0:
        raise RuntimeError(f"{split_name} evaluation produced no supervised tokens")
    mean_loss = total_loss_sum / supervised_count
    surprise_array = np.asarray(surprise_values, dtype=np.float64)
    load_report = expert_load_summary(expert_counts)
    result = {
        "split": split_name,
        "batches": len(batch_times),
        "loss_nats": mean_loss,
        "task_loss_nats": task_loss_sum / supervised_count,
        "bpb": mean_loss / math.log(2),
        "perplexity": math.exp(min(80.0, mean_loss)),
        "answer_loss_nats": answer_loss_sum / answer_count if answer_count else None,
        "answer_bpb": answer_loss_sum / answer_count / math.log(2) if answer_count else None,
        "thinking_loss_nats": thinking_loss_sum / thinking_count if thinking_count else None,
        "thinking_bpb": thinking_loss_sum / thinking_count / math.log(2) if thinking_count else None,
        "supervised_tokens": supervised_count,
        "answer_tokens": answer_count,
        "thinking_tokens": thinking_count,
        "valid_input_tokens": valid_count,
        "forward_tokens_per_second": valid_count / max(sum(batch_times), 1e-9),
        "batch_seconds_p50": float(np.percentile(batch_times, 50)),
        "batch_seconds_p95": float(np.percentile(batch_times, 95)),
        "loss_standard_error": online_statistics(total_loss_sum, loss_square_sum, supervised_count)["standard_error"],
        "answer_loss_standard_error": online_statistics(answer_loss_sum, answer_loss_square_sum, answer_count)["standard_error"],
        "surprise_mean": float(surprise_array.mean()) if surprise_array.size else None,
        "surprise_p50": float(np.percentile(surprise_array, 50)) if surprise_array.size else None,
        "surprise_p95": float(np.percentile(surprise_array, 95)) if surprise_array.size else None,
        "state_norm_means": {name: float(np.mean(values)) for name, values in state_norms.items()},
        "nonfinite_batches": nonfinite_batches,
        "gpu_peak_allocated_bytes": torch.cuda.max_memory_allocated(DEVICE),
        "expert_load": load_report,
        "expected_active_assignments": valid_count * MODEL_SETTINGS.expert_top_k,
        "active_assignment_count_matches": load_report["total_assignments"] == valid_count * MODEL_SETTINGS.expert_top_k,
    }
    model.train()
    return result

torch.cuda.reset_peak_memory_stats(DEVICE)
validation_report = evaluate_loader(validation_loader, "validation")
torch.cuda.reset_peak_memory_stats(DEVICE)
training_sample_report = evaluate_loader(train_loader, "training_sample", maximum_batches=256)
EVALUATION_REPORT = {"validation": validation_report, "training_sample": training_sample_report}
(RESULTS_DIR / "evaluation.json").write_text(json.dumps(EVALUATION_REPORT, indent=2), encoding="utf-8")
print(json.dumps(EVALUATION_REPORT, indent=2))

In [ ]:
def benchmark_forward(candidate_model, label, batch_size, sequence_length, warmup_runs=3, timed_runs=12):
    candidate_model.eval()
    inputs = torch.randint(0, 256, (batch_size, sequence_length), device=DEVICE)
    with torch.inference_mode():
        for _ in range(warmup_runs):
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                candidate_model(inputs, execution_mode=ExecutionMode.PARALLEL)
        synchronize_cuda()
        torch.cuda.reset_peak_memory_stats(DEVICE)
        timings = []
        for _ in range(timed_runs):
            synchronize_cuda()
            started_at = time.perf_counter()
            with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                candidate_model(inputs, execution_mode=ExecutionMode.PARALLEL)
            synchronize_cuda()
            timings.append(time.perf_counter() - started_at)
    return {
        "label": label,
        "batch_size": batch_size,
        "sequence_length": sequence_length,
        "warmup_runs": warmup_runs,
        "timed_runs": timed_runs,
        "batch_seconds_p50": float(np.percentile(timings, 50)),
        "batch_seconds_p95": float(np.percentile(timings, 95)),
        "tokens_per_second_p50": batch_size * sequence_length / np.percentile(timings, 50),
        "tokens_per_second_p95_window": batch_size * sequence_length / np.percentile(timings, 95),
        "gpu_peak_allocated_bytes": torch.cuda.max_memory_allocated(DEVICE),
        "parameters": parameter_summary(candidate_model)["parameters"],
    }

benchmark_batch_size = min(BATCH_SIZE, 4)
throughput_results = []
for sequence_length in (64, 128, 256):
    throughput_results.append(
        benchmark_forward(model, "trained_128_experts_top_6", benchmark_batch_size, sequence_length)
    )
control_model = KoemiModel(
    ModelSettings(
        embedding_size=MODEL_SETTINGS.embedding_size,
        memory_features=MODEL_SETTINGS.memory_features,
        local_memory_size=MODEL_SETTINGS.local_memory_size,
        salience_memory_size=MODEL_SETTINGS.salience_memory_size,
        salience_threshold=MODEL_SETTINGS.salience_threshold,
        expert_count=0,
        expert_top_k=1,
        cache_capacity=MODEL_SETTINGS.cache_capacity,
        scan_chunk=MODEL_SETTINGS.scan_chunk,
        refine_decay_rate=MODEL_SETTINGS.refine_decay_rate,
        ablation=MODEL_SETTINGS.ablation,
    )
).to(DEVICE)
for sequence_length in (64, 128, 256):
    throughput_results.append(
        benchmark_forward(control_model, "no_expert_speed_control_untrained", benchmark_batch_size, sequence_length)
    )
THROUGHPUT_REPORT = {
    "benchmarks": throughput_results,
    "interpretation": "The no-expert control is a runtime comparison, not a quality baseline; its random weights are not evaluated as a model.",
}
(RESULTS_DIR / "throughput.json").write_text(json.dumps(THROUGHPUT_REPORT, indent=2), encoding="utf-8")
print(json.dumps(THROUGHPUT_REPORT, indent=2))
del control_model
torch.cuda.empty_cache()

In [ ]:
from koemi.model.cache import DiskMappingCache, WarmTokenCache
from koemi.data.tokenizer import ByteTokenizer
from koemi.training.generation import evaluate_prompt_state, generate_text

cache_directory = RESULTS_DIR / "prefix-cache"
mapping_cache = DiskMappingCache(
    cache_directory,
    capacity=64,
    namespace=f"koemi-3hip-t4-{SEED}",
    ttl_seconds=24 * 60 * 60,
)
mapping_cache.clear()
prefix_bytes = (b"Koemi HERM prefix ledger measurement. " * 16)[:256]
extension_bytes = prefix_bytes + b" This is a new suffix."
prefix_tensor = torch.tensor([list(prefix_bytes)], dtype=torch.long, device=DEVICE)
extension_tensor = torch.tensor([list(extension_bytes)], dtype=torch.long, device=DEVICE)
model.eval()
with torch.inference_mode():
    synchronize_cuda()
    first_started_at = time.perf_counter()
    first_evaluation = evaluate_prompt_state(model, prefix_tensor, None, mapping_cache)
    synchronize_cuda()
    first_seconds = time.perf_counter() - first_started_at
    synchronize_cuda()
    second_started_at = time.perf_counter()
    second_evaluation = evaluate_prompt_state(model, extension_tensor, None, mapping_cache)
    synchronize_cuda()
    second_seconds = time.perf_counter() - second_started_at
    oracle_output = model(extension_tensor, execution_mode=ExecutionMode.PARALLEL)
warm_cache = WarmTokenCache(MODEL_SETTINGS.cache_capacity)
with torch.inference_mode():
    model(prefix_tensor[:, :128], warm_cache=warm_cache)
    model(prefix_tensor[:, :128], warm_cache=warm_cache)
warm_statistics = asdict(warm_cache.statistics())
prefix_statistics = asdict(mapping_cache.statistics())
PREFIX_CACHE_REPORT = {
    "prefix_bytes": len(prefix_bytes),
    "extension_bytes": len(extension_bytes),
    "first_processed_tokens": first_evaluation.processed_tokens,
    "second_processed_tokens": second_evaluation.processed_tokens,
    "second_reused_prefix_tokens": second_evaluation.reused_prefix_tokens,
    "first_seconds": first_seconds,
    "second_seconds": second_seconds,
    "oracle_max_logit_error": float(
        (second_evaluation.last_logits - oracle_output.logits[:, -1]).abs().max().cpu()
    ),
    "prefix_state_bytes": MODEL_REPORT["state_at_capacity_probe"]["total_bytes"],
    "mapping_cache_statistics": prefix_statistics,
    "warm_token_cache_statistics": warm_statistics,
}
(RESULTS_DIR / "cache_probe.json").write_text(json.dumps(PREFIX_CACHE_REPORT, indent=2), encoding="utf-8")
tokenizer = ByteTokenizer()
generated_text = generate_text(
    model,
    tokenizer,
    "Explain why a queue preserves arrival order.",
    max_new_bytes=128,
    temperature=0.8,
    device=DEVICE,
)
(RESULTS_DIR / "sample_generation.txt").write_text(generated_text, encoding="utf-8")
print(json.dumps(PREFIX_CACHE_REPORT, indent=2))
print({"sample_generation_path": str(RESULTS_DIR / "sample_generation.txt")})

In [ ]:
from koemi.training.checkpoints import CheckpointStore
CLI_CHECKPOINT_PATH = RESULTS_DIR / "koemi-3hip-t4-moe-model.pt"
CheckpointStore().save(CLI_CHECKPOINT_PATH, model, overwrite=True)

def read_json_lines(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

step_history = read_json_lines(STEP_LOG_PATH)
epoch_history = read_json_lines(EPOCH_LOG_PATH)
report = {
    "experiment": "Koemi-3HIP HERM T4 overnight",
    "status": "measured on this Colab run",
    "environment": ENVIRONMENT,
    "dataset": DATASET_REPORT,
    "model": MODEL_REPORT,
    "preflight": PREFLIGHT_REPORT,
    "training_session": TRAINING_SESSION,
    "evaluation": EVALUATION_REPORT,
    "throughput": THROUGHPUT_REPORT,
    "cache_probe": PREFIX_CACHE_REPORT,
    "cli_checkpoint": str(CLI_CHECKPOINT_PATH),
    "formulas": {
        "causal_loss_nats": "-(1/N) * sum(log p(target_byte | previous_state))",
        "bpb": "loss_nats / ln(2)",
        "perplexity": "exp(loss_nats)",
        "surprise": "1 - exp(-NLL / ln(256))",
        "active_assignments": "valid_input_tokens * expert_top_k",
        "expert_entropy": "-sum(p_e * ln(p_e))",
        "expert_gini": "sum_e,f |load_e-load_f| / (2 * experts * total_load)",
    },
    "documentation_read": [
        "README.md",
        "docs/KOEMI_ARCHITECTURE.md",
        "docs/BENCHMARK.md",
        "MapSource.md",
        "src/koemi/configuration/settings.py",
        "src/koemi/data/adapters.py",
        "src/koemi/data/serialization.py",
        "src/koemi/training/dataset.py",
        "src/koemi/training/objective.py",
        "src/koemi/training/trainer.py",
        "src/koemi/model/memory.py",
        "src/koemi/model/network.py",
        "src/koemi/model/cache.py",
        "src/koemi/training/generation.py",
    ],
}
(RESULTS_DIR / "final_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")

plt.rcParams.update(
    {
        "font.family": "DejaVu Sans",
        "text.color": "#202020",
        "axes.labelcolor": "#202020",
        "xtick.color": "#202020",
        "ytick.color": "#202020",
        "axes.edgecolor": "#6b5a00",
    }
)
figure, axes = plt.subplots(2, 2, figsize=(15, 9), facecolor="white")
if step_history:
    step_frame = pd.DataFrame(step_history)
    axes[0, 0].plot(step_frame["step"], step_frame["bpb"], color="#9c7c00", linewidth=1.2)
    axes[0, 0].set_title("Training BPB")
    axes[0, 0].set_xlabel("optimizer step")
    axes[0, 0].set_ylabel("bits per byte")
    axes[0, 1].plot(step_frame["step"], step_frame["tokens_per_second"], color="#6f5b00", linewidth=1.0)
    axes[0, 1].set_title("Training throughput")
    axes[0, 1].set_xlabel("optimizer step")
    axes[0, 1].set_ylabel("supervised tokens/s")
    axes[1, 0].plot(step_frame["step"], step_frame["gpu_peak_allocated_bytes"] / 2**30, color="#c5a900", linewidth=1.0)
    axes[1, 0].set_title("Peak allocated GPU memory")
    axes[1, 0].set_xlabel("optimizer step")
    axes[1, 0].set_ylabel("GiB")
expert_load = np.asarray(validation_report["expert_load"]["counts"])
axes[1, 1].bar(np.arange(MODEL_SETTINGS.expert_count), expert_load, color="#e8d98a", edgecolor="#806800", linewidth=0.2)
axes[1, 1].set_title("Validation active assignment load")
axes[1, 1].set_xlabel("expert index")
axes[1, 1].set_ylabel("assignments")
for axis in axes.flat:
    axis.set_facecolor("white")
    axis.grid(axis="y", color="#eee6b5", linewidth=0.6)
figure.tight_layout()
figure.savefig(RESULTS_DIR / "evidence.png", dpi=180, facecolor="white")
plt.show()

archive_base = str(RESULTS_DIR.parent / RESULTS_DIR.name)
archive_path = shutil.make_archive(archive_base, "zip", RESULTS_DIR)
print(json.dumps({
    "final_report": str(RESULTS_DIR / "final_report.json"),
    "evidence_plot": str(RESULTS_DIR / "evidence.png"),
    "checkpoint": str(CHECKPOINT_PATH),
    "cli_checkpoint": str(CLI_CHECKPOINT_PATH),
    "zip_archive": archive_path,
    "training_steps_logged": len(step_history),
    "training_epochs_logged": len(epoch_history),
}, indent=2))

## Reading the final report

The model predicts bytes causally. If `y_t` is the supervised target byte and `p_t(y_t)` is its predicted probability, the repository's loss is `L = -(1/N) Σ log p_t(y_t)`. The notebook reports `BPB = L / ln(2)` and `perplexity = exp(L)`. `answer_bpb` uses only answer bytes; `thinking_bpb` uses only thinking bytes. This prevents a different span mixture from manufacturing a quality improvement.

HERM's recurrent state is fixed-width. The state contains the working vector, fast and slow associative matrices and normalizers, exact local slots, exact surprise-admitted slots and the previous byte. Its byte size is printed from tensor shapes and dtypes; it does not grow with the number of processed tokens. The local and salient rings are causal: a position can see only carried entries and earlier positions.

The associative read is `raw = (B ψ)/(c·ψ + ε)` with `confidence = (c·ψ)/(c·ψ + ε)` and `memory_read = confidence * raw`. The preflight test verifies that zero evidence returns zero. The parallel path contracts rank-one writes through a causal `[B,C,C]` influence matrix; the preflight compares its logits, state and expert assignments against the sequential oracle.

Expert load is counted over valid input positions, not supervised targets. Therefore the invariant is `total_active_assignments = valid_input_tokens × 6`. Entropy and Gini describe balance only; they do not prove that experts learned useful semantic roles.

This notebook can establish a reproducible result for this HERM configuration on one T4 run. It cannot establish Transformer parity, arbitrary long-context recall, learned routing quality, or production reliability. Those require the matched-tokenizer, matched-budget, multi-seed comparisons specified in `docs/BENCHMARK.md`.